# 实验结果汇总表
本 Notebook 自动读取 `ranking` 和 `aft` 文件夹下的实验结果，并一键生成按不同对照组分类的 Markdown 表格。

- **Ranking Base**: m=10, n=100, pc=0.3
- **AFT Base**: m=10, n=100, pc=0.3, cens=0.2

运行下方代码块即可生成所有表格结果。

In [ ]:
import os
import json
import numpy as np
import glob
from collections import defaultdict
from IPython.display import display, Markdown

def parse_filename(f):
    basename = os.path.basename(f)
    parts = basename.replace('.json', '').split('_')
    parts[-1] = parts[-1].split(' ')[0] # handle ' (1)'
    
    info = {'noise': parts[0]}
    for p in parts[1:]:
        if p.startswith('m'): info['m'] = p[1:]
        elif p.startswith('n') and not p.startswith('normal') and not p.startswith('noise'): info['n'] = p[1:]
        elif p.startswith('pc'): info['pc'] = p[2] + '.' + p[3:] if len(p)>3 else p[2:]
        elif p.startswith('cens'): info['cens'] = p[4] + '.' + p[5:] if len(p)>5 else p[4:]
    return info

def get_setting_name(info, task):
    m = info.get('m')
    n = info.get('n')
    pc = info.get('pc')
    cens = info.get('cens', '0.2') # default to 0.2 if not found
    
    if task == 'ranking':
        if m == '10' and n == '100' and pc == '0.3': return 'Base'
        if m == '20' and n == '100' and pc == '0.3': return 'm=20'
        if m == '10' and n == '200' and pc == '0.3': return 'n=200'
        if m == '10' and n == '100' and pc == '0.5': return 'pc=0.5'
        return 'Base'
    else:
        # For AFT, Base is m=10, n=100, pc=0.3, cens=0.2
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'Base'
        if m == '20' and n == '100' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'm=20'
        if m == '10' and n == '200' and pc == '0.3' and (cens == '0.2' or cens == '02'): return 'n=200'
        if m == '10' and n == '100' and pc == '0.5' and (cens == '0.2' or cens == '02'): return 'pc=0.5'
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.3' or cens == '03'): return 'cens=0.3'
        if m == '10' and n == '100' and pc == '0.3' and (cens == '0.4' or cens == '04'): return 'cens=0.4'
        return 'Base'

def generate_table(task, folder):
    if not os.path.exists(folder): return "Folder not found."
    files = glob.glob(os.path.join(folder, '*.json'))
    data_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    methods = ['Global', 'Local', 'Avg', 'D-ProxGD', 'U-ADMM']
    
    for f in files:
        info = parse_filename(f)
        noise = info['noise']
        setting = get_setting_name(info, task)
        
        with open(f, 'r', encoding='utf-8') as file:
            try:
                data = json.load(file)
                for r in data.get('results', []):
                    for m in methods:
                        if m in r and 'RMSE' in r[m]:
                            data_dict[noise][setting][m].append(r[m]['RMSE'])
            except:
                pass
                
    if task == 'ranking':
        cols = ['Base', 'm=20', 'n=200', 'pc=0.5']
    else:
        cols = ['Base', 'm=20', 'n=200', 'pc=0.5', 'cens=0.3', 'cens=0.4']
        
    md_output = ""
    for noise in sorted(data_dict.keys()):
        md_output += f"### Task: {task.capitalize()} | Noise: {noise}\n\n"
        header = "| Method | " + " | ".join(cols) + " |"
        separator = "|---" + "|---" * len(cols) + "|"
        md_output += header + "\n" + separator + "\n"
        
        for m in methods:
            row = f"| {m} | "
            vals = []
            for c in cols:
                rmses = data_dict[noise][c][m]
                if len(rmses) > 0:
                    vals.append(f"{np.mean(rmses):.4f}")
                else:
                    vals.append("-")
            row += " | ".join(vals) + " |"
            md_output += row + "\n"
        md_output += "\n"
    return md_output

md_ranking = generate_table('ranking', 'ranking')
md_aft = generate_table('aft', 'aft')

display(Markdown("# Ranking 实验汇总"))
display(Markdown(md_ranking))
display(Markdown("# AFT 实验汇总"))
display(Markdown(md_aft))
